<a href="https://colab.research.google.com/github/AnanyaAsthana/Hadoop-CUDA-Lab/blob/main/Ques1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile bfs_cuda.cu
#include <stdio.h>
#include <cuda.h>

#define V 4

__global__ void bfs_kernel(int *adj, int *frontier, int *next_frontier,
                           int *visited, int *level, int curr_level) {

    int u = blockIdx.x * blockDim.x + threadIdx.x;

    if (u >= V) return;

    if (frontier[u] == 1) {

        for (int v = 0; v < V; v++) {

            if (adj[u * V + v] == 1) {

                if (atomicCAS(&visited[v], 0, 1) == 0) {
                    level[v] = curr_level + 1;
                    next_frontier[v] = 1;
                }
            }
        }
    }
}


int main() {

    int adj[V][V] = {
        {0,1,1,0},
        {1,0,0,1},
        {1,0,0,1},
        {0,1,1,0},
    };

    int source = 0;

    int frontier[V] = {0};
    int next_frontier[V] = {0};
    int visited[V] = {0};
    int level[V];

    for (int i = 0; i < V; i++) {
        level[i] = -1;
    }

    frontier[source] = 1;
    visited[source] = 1;
    level[source] = 0;

    int *d_adj, *d_frontier, *d_next_frontier, *d_visited, *d_level;

    cudaMalloc(&d_adj, V * V * sizeof(int));
    cudaMalloc(&d_frontier, V * sizeof(int));
    cudaMalloc(&d_next_frontier, V * sizeof(int));
    cudaMalloc(&d_visited, V * sizeof(int));
    cudaMalloc(&d_level, V * sizeof(int));

    cudaMemcpy(d_adj, adj, V * V * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_frontier, frontier, V * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_visited, visited, V * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_level, level, V * sizeof(int), cudaMemcpyHostToDevice);

    int curr_level = 0;

    while (1) {

        cudaMemset(d_next_frontier, 0, V * sizeof(int));

        bfs_kernel<<<1, V>>>(d_adj, d_frontier, d_next_frontier,
                             d_visited, d_level, curr_level);

        cudaMemcpy(next_frontier, d_next_frontier, V * sizeof(int), cudaMemcpyDeviceToHost);

        int done = 1;

        for (int i = 0; i < V; i++) {
            if (next_frontier[i] == 1) {
                done = 0;
                frontier[i] = 1;
            } else {
                frontier[i] = 0;
            }
        }

        cudaMemcpy(d_frontier, frontier, V * sizeof(int), cudaMemcpyHostToDevice);

        if (done) break;

        curr_level++;
    }

    cudaMemcpy(level, d_level, V * sizeof(int), cudaMemcpyDeviceToHost);

    printf("\nVertex Levels from Source %d:\n", source);
    for (int i = 0; i < V; i++) {
        printf("Vertex %d -> Level %d\n", i, level[i]);
    }

    cudaFree(d_adj);
    cudaFree(d_frontier);
    cudaFree(d_next_frontier);
    cudaFree(d_visited);
    cudaFree(d_level);

    return 0;
}

Overwriting bfs_cuda.cu


In [ ]:
!nvcc bfs_cuda.cu -o bfs

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./bfs


Vertex Levels from Source 0:
Vertex 0 -> Level 0
Vertex 1 -> Level 1
Vertex 2 -> Level 1
Vertex 3 -> Level 2
